# ResearchLanka Kaggle Main-Branch Full Run

Import this notebook into Kaggle and run cells from top to bottom.

It will:

- clone/pull the latest `main` branch
- copy your uploaded raw dataset into the repo
- install Python + Dagster dependencies
- run the Dagster no-collection preprocessing job
- build best-quality embeddings
- train Logistic Regression
- train Multinomial Naive Bayes baseline
- train Linear SVM
- run formal model comparison and promote the best flat classifier
- train hierarchical field -> subfield Linear SVM
- run NMF topic modeling and evaluation
- zip outputs for download

It does **not** collect data from APIs or repositories.


## 1. Settings

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/krish-anu/researchlanka-ai.git"
BRANCH = "main"
WORK_DIR = Path("/kaggle/working")
CODE_DIR = WORK_DIR / "code"
BACKEND_DIR = CODE_DIR / "backend"
DATASET_DATA_DIR = Path("/kaggle/input/datasets/anusankrishnathas/raw-data1/backend/data")
OUTPUT_ZIP = WORK_DIR / "researchlanka-kaggle-outputs.zip"

print("Repo:", REPO_URL)
print("Branch:", BRANCH)
print("Code dir:", CODE_DIR)
print("Backend dir:", BACKEND_DIR)
print("Dataset data dir:", DATASET_DATA_DIR)
print("Output zip:", OUTPUT_ZIP)

Repo: https://github.com/krish-anu/researchlanka-ai.git
Branch: main
Code dir: /kaggle/working/code
Backend dir: /kaggle/working/code/backend
Dataset data dir: /kaggle/input/datasets/anusankrishnathas/raw-data1/backend/data
Output zip: /kaggle/working/researchlanka-kaggle-outputs.zip


## 2. Check Kaggle Dataset Exists

If this fails, your Kaggle dataset path is different. Update `DATASET_DATA_DIR` above.

In [2]:
!ls -la /kaggle/input
!find /kaggle/input -maxdepth 5 -type d | head -80
!test -d {DATASET_DATA_DIR} && echo "Dataset path OK" || echo "Dataset path NOT FOUND"

total 12
drwxr-xr-x 3 root root 4096 Aug 31 04:08 .
drwxr-xr-x 8 root root 4096 Aug 31 04:08 ..
drwxr-xr-x 3 root root 4096 Aug 31 04:08 datasets
/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/anusankrishnathas
/kaggle/input/datasets/anusankrishnathas/raw-data1
/kaggle/input/datasets/anusankrishnathas/raw-data1/backend
/kaggle/input/datasets/anusankrishnathas/raw-data1/backend/data
Dataset path OK


## 3. Clone Or Pull Latest Main Branch

In [3]:
%cd /kaggle/working
if CODE_DIR.exists() and (CODE_DIR / ".git").exists():
    %cd /kaggle/working/code
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} code
    %cd /kaggle/working/code

!git log --oneline -3
!ls

/kaggle/working
Cloning into 'code'...
remote: Enumerating objects: 3530, done.
remote: Counting objects: 100% (333/333), done.
remote: Compressing objects: 100% (238/238), done.
remote: Total 3530 (delta 119), reused 134 (delta 94), pack-reused 3197 (from 2)
Receiving objects: 100% (3530/3530), 13.88 MiB | 13.21 MiB/s, done.
Resolving deltas: 100% (2102/2102), done.
/kaggle/working/code
e6d616f (HEAD -> main, origin/main, origin/HEAD) Merge pull request #490 from krish-anu/feature/machine-learning
97e8f06 (origin/feature/machine-learning) Fix Kaggle Dagster notebook import path
12dd3d1 Merge pull request #489 from krish-anu/feature/machine-learning
backend       docs		 frontend	   Makefile   README.md
CHANGELOG.md  dse-project.ipynb  KAGGLE_README.md  notebooks  scripts


## 4. Copy Uploaded Raw Data Into Backend

In [4]:
%cd /kaggle/working/code/backend
!rm -rf data
!mkdir -p data
!cp -r {DATASET_DATA_DIR}/* data/
!find data -maxdepth 3 -type f | head -60

/kaggle/working/code/backend
data/config/repositories.json
data/reports/dagster_collection_summary_20260804T085133Z.json
data/processed/repositories/pdn.jsonl
data/processed/repositories/ou.jsonl
data/processed/repositories/nsf.jsonl
data/processed/repositories/uom.jsonl
data/processed/repositories/seu.jsonl
data/processed/repositories/sliit.jsonl
data/processed/repositories/jfn_medicine.jsonl
data/processed/repositories/busl.jsonl
data/processed/repositories/cmb.jsonl
data/processed/repositories/ruh.jsonl
data/processed/repositories/jfn_research.jsonl
data/processed/sljol.csv
data/processed/repositories_combined.csv
data/processed/crossref/crossref_sri_lanka_works.csv
data/processed/crossref/crossref_sri_lanka_works.jsonl
data/raw/cmb/rest_items.jsonl
data/raw/vpa/oai_dc.jsonl
data/raw/rjt/oai_dc.jsonl
data/raw/vau/oai_dc.jsonl
data/raw/busl/rest_items.jsonl
data/raw/pdn/rest_items.jsonl
data/raw/jfn_medicine/html_meta.jsonl
data/raw/jfn_research/html_meta.jsonl
data/raw/sljol/crossre

## 5. Install Dependencies

Kaggle may show dependency conflict warnings. Continue if the install completes. If the session restarts, rerun from the top.

In [5]:
%cd /kaggle/working/code/backend

!python -m pip install -r requirements.txt "protobuf<6" "google-cloud-bigquery-storage>=2.30,<3"

!python -m pip install "dagster==1.13.16" "dagster-webserver==1.13.16" "protobuf<6" "google-cloud-bigquery-storage>=2.30,<3"

!python -m pip install -e . --no-deps

!python -m pip install -e dagster-quickstart --no-deps

!python -m dagster --version

import pandas as pd
import sklearn
import pyarrow as pa
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("pyarrow", pa.__version__)


/kaggle/working/code/backend
Ignoring numpy: markers 'python_version >= "3.14"' don't match your environment
Ignoring pandas: markers 'python_version >= "3.14"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of google-cloud-bigquery-storage to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.3/224.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.4/117.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 37.4 MB/s eta 0:00:00
   ━━━━

## 6. Run Dagster Pipeline Without Data Collection

This prepares existing source files and runs preprocessing through the analysis-ready dataset.

In [6]:
%cd /kaggle/working/code/backend/dagster-quickstart

import importlib
import sys
from pathlib import Path

BACKEND_DIR = Path("/kaggle/working/code/backend")
DAGSTER_SRC_DIR = BACKEND_DIR / "dagster-quickstart" / "src"
for path in (DAGSTER_SRC_DIR, BACKEND_DIR):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)
importlib.invalidate_caches()

from dagster_quickstart.definitions import defs

loaded_defs = defs() if callable(defs) else defs
job = loaded_defs.resolve_job_def("researchlanka_no_collection_preprocessing_job")
result = job.execute_in_process()
if not result.success:
    raise RuntimeError("Dagster preprocessing job failed")


/kaggle/working/code/backend/dagster-quickstart


2026-08-31 04:10:29 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 6dc875ea-f0f2-4a7f-bbb1-40587db20b17 - 16 - RUN_START - Started execution of run for "researchlanka_no_collection_preprocessing_job".
2026-08-31 04:10:29 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 6dc875ea-f0f2-4a7f-bbb1-40587db20b17 - 16 - ENGINE_EVENT - Executing steps in process (pid: 16)
2026-08-31 04:10:29 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 6dc875ea-f0f2-4a7f-bbb1-40587db20b17 - 16 - RESOURCE_INIT_STARTED - Starting initialization of resources [io_manager].
2026-08-31 04:10:30 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 6dc875ea-f0f2-4a7f-bbb1-40587db20b17 - 16 - RESOURCE_INIT_SUCCESS - Finished initialization of resources [io_manager].
2026-08-31 04:10:30 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 6dc875ea-f0f2-4a7f-bbb1-40587db20b17 - 16 - LOGS_CAPTURED - Ca

busl: mapped 2885 records via rest -> /kaggle/working/code/backend/data/processed/repositories/busl.jsonl
cmb: mapped 8456 records via rest -> /kaggle/working/code/backend/data/processed/repositories/cmb.jsonl
esn: mapped 0 records (raw files empty)
jfn_medicine: mapped 3766 records via html -> /kaggle/working/code/backend/data/processed/repositories/jfn_medicine.jsonl
jfn_research: mapped 11106 records via html -> /kaggle/working/code/backend/data/processed/repositories/jfn_research.jsonl
nsf: mapped 15792 records via rest -> /kaggle/working/code/backend/data/processed/repositories/nsf.jsonl
ou: mapped 3627 records via oai -> /kaggle/working/code/backend/data/processed/repositories/ou.jsonl
pdn: mapped 7740 records via rest -> /kaggle/working/code/backend/data/processed/repositories/pdn.jsonl
rjt: mapped 0 records (raw files empty)
ruh: mapped 0 records (raw files empty)
seu: mapped 6606 records via oai -> /kaggle/working/code/backend/data/processed/repositories/seu.jsonl
sliit: mappe

2026-08-31 04:10:50 +0000 - dagster - INFO - researchlanka_no_collection_preprocessing_job - 6dc875ea-f0f2-4a7f-bbb1-40587db20b17 - researchlanka_all_sources_collected - Converting 11 repository JSONL files to CSV: /kaggle/working/code/backend/data/processed/repositories_combined.csv.


uom: mapped 16540 records via oai -> /kaggle/working/code/backend/data/processed/repositories/uom.jsonl
uwu: mapped 0 records (raw files empty)
vau: mapped 0 records (raw files empty)
vpa: mapped 0 records (raw files empty)


2026-08-31 04:11:00 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 6dc875ea-f0f2-4a7f-bbb1-40587db20b17 - 16 - researchlanka_all_sources_collected - STEP_OUTPUT - Yielded output "result" of type "Dict[String,Any]". (Type check passed).
2026-08-31 04:11:00 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 6dc875ea-f0f2-4a7f-bbb1-40587db20b17 - researchlanka_all_sources_collected - Writing file at: /tmp/tmp06qw6xuy/storage/researchlanka_all_sources_collected using PickledObjectFilesystemIOManager...
2026-08-31 04:11:00 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 6dc875ea-f0f2-4a7f-bbb1-40587db20b17 - 16 - researchlanka_all_sources_collected - ASSET_MATERIALIZATION - Materialized value researchlanka_all_sources_collected.
2026-08-31 04:11:00 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 6dc875ea-f0f2-4a7f-bbb1-40587db20b17 - 16 - researchlanka_all_sources_collected - HANDLED_OUTPUT

In [7]:
%cd /kaggle/working/code/backend
!make openalex-lk-audit PYTHON=python

from pathlib import Path
import json
import pandas as pd
from IPython.display import FileLink, display

AUDIT_DIR = Path("data/reports/openalex_lk_affiliation_audit")
SUMMARY_PATH = AUDIT_DIR / "lk_affiliation_audit_summary.json"
REPORT_PATH = AUDIT_DIR / "lk_affiliation_audit_report.md"

if not SUMMARY_PATH.exists():
    raise FileNotFoundError(f"Missing audit summary: {SUMMARY_PATH}")

summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
overall = summary["overall"]
impact = summary["publication_impact"]
problems = summary["potential_problems"]

print("OpenAlex LK affiliation audit complete")
print("Target: verified publication-time Sri Lankan institutional authorship")
print(f"Report: {REPORT_PATH}")
print(f"Summary JSON: {SUMMARY_PATH}")
print()

metrics = pd.DataFrame([
    {"metric": "Unique OpenAlex works", "value": f"{overall['unique_openalex_work_ids']:,}"},
    {"metric": "Total authorships", "value": f"{overall['total_authorships']:,}"},
    {"metric": "Currently LK authorships", "value": f"{overall['currently_lk_authorships']:,}"},
    {"metric": "Strict verified works", "value": f"{impact['strict_verified_dataset_size']:,}"},
    {"metric": "Retained under strict rule", "value": f"{impact['percentage_retained']}%"},
    {"metric": "Works sent to review", "value": f"{impact['records_sent_to_review']:,}"},
    {"metric": "LK authorships with any issue", "value": f"{problems['at_least_one_issue_authorships']['count']:,} ({problems['at_least_one_issue_authorships']['percent']}%)"},
    {"metric": "Normalized-LK-only authorships", "value": f"{problems['normalized_lk_only_authorships']['count']:,} ({problems['normalized_lk_only_authorships']['percent']}%)"},
    {"metric": "Explicit foreign-location conflicts", "value": f"{problems['explicit_country_conflict_authorships']['count']:,} ({problems['explicit_country_conflict_authorships']['percent']}%)"},
])
display(metrics)

print("Audit files")
for file_name in [
    "lk_affiliation_audit_report.md",
    "lk_affiliation_audit_summary.json",
    "lk_affiliation_audit_records.csv",
    "lk_affiliation_manual_review.csv",
    "verified_lk_authorships.csv",
]:
    file_path = AUDIT_DIR / file_name
    size_mb = file_path.stat().st_size / (1024 * 1024) if file_path.exists() else 0
    print(f"- {file_path} ({size_mb:.2f} MB)")

print()
print("Report preview")
print("=" * 80)
print("\n".join(REPORT_PATH.read_text(encoding="utf-8").splitlines()[:80]))
print("=" * 80)

display(FileLink(str(REPORT_PATH)))
display(FileLink(str(SUMMARY_PATH)))


/kaggle/working/code/backend
python -m src.quality.audit_openalex_lk_affiliations \
--input data/raw/openalex/openalex_sri_lanka_works.jsonl \
--output-dir data/reports/openalex_lk_affiliation_audit \
--log-level INFO \

2026-08-31 04:30:42,483 INFO openalex_lk_affiliation_audit: Building local author history from data/raw/openalex/openalex_sri_lanka_works.jsonl
2026-08-31 04:30:46,535 INFO openalex_lk_affiliation_audit: Pass 1 read 10000 works
2026-08-31 04:30:50,057 INFO openalex_lk_affiliation_audit: Pass 1 read 20000 works
2026-08-31 04:30:53,640 INFO openalex_lk_affiliation_audit: Pass 1 read 30000 works
2026-08-31 04:30:56,479 INFO openalex_lk_affiliation_audit: Pass 1 read 40000 works
2026-08-31 04:30:59,600 INFO openalex_lk_affiliation_audit: Pass 1 read 50000 works
2026-08-31 04:31:00,673 INFO openalex_lk_affiliation_audit: Author history contains 81184 authors
2026-08-31 04:31:17,692 INFO openalex_lk_affiliation_audit: Pass 2 audited 5000 works
2026-08-31 04:31:31,925 INFO op

,metric,value
0,Unique OpenAlex works,"53,672"
1,Total authorships,"184,485"
2,Currently LK authorships,"156,473"
3,Strict verified works,"47,102"
4,Retained under strict rule,87.759%
5,Works sent to review,"6,082"
6,LK authorships with any issue,"17,886 (11.4307%)"
7,Normalized-LK-only authorships,"10,666 (6.8165%)"
8,Explicit foreign-location conflicts,"5,410 (3.4575%)"


Audit files
- data/reports/openalex_lk_affiliation_audit/lk_affiliation_audit_report.md (0.03 MB)
- data/reports/openalex_lk_affiliation_audit/lk_affiliation_audit_summary.json (0.06 MB)
- data/reports/openalex_lk_affiliation_audit/lk_affiliation_audit_records.csv (15.88 MB)
- data/reports/openalex_lk_affiliation_audit/lk_affiliation_manual_review.csv (9.92 MB)
- data/reports/openalex_lk_affiliation_audit/verified_lk_authorships.csv (96.09 MB)

Report preview
# OpenAlex LK Affiliation Audit

Target concept: verified publication-time Sri Lankan institutional authorship. The audit does not infer nationality.

## Overall
- Total works: 53,672
- Unique OpenAlex work IDs: 53,672
- Total authorships: 184,485
- Unique authors: 81,184
- Currently LK authorships: 156,473
- Works passing current `keep_in_sri_lanka_owned_dataset`: 53,672

## Publication Impact
- Current dataset size: 53,672 works
- Strict verified dataset size: 47,102 works
- Records removed: 6,570
- Records sent to review: 6,082

/kaggle/working/code/backend/data/reports/openalex_lk_affiliation_audit/lk_affiliation_audit_report.md

/kaggle/working/code/backend/data/reports/openalex_lk_affiliation_audit/lk_affiliation_audit_summary.json

## 7. Verify Preprocessing Outputs

In [8]:
%cd /kaggle/working/code/backend
!ls -lh data/processed/repositories_combined.csv
!ls -lh data/processed/sljol.csv
!ls -lh data/processed/common/common_publications_final.csv
!ls -lh data/processed/common/common_publications_final_2016_2026_analysis_ready.csv

import pandas as pd
paths = [
    'data/processed/common/common_publications_final.csv',
    'data/processed/common/common_publications_final_2016_2026_analysis_ready.csv',
]
for path in paths:
    frame = pd.read_csv(path, nrows=5)
    total = sum(1 for _ in open(path, encoding='utf-8')) - 1
    print(path, 'rows=', total, 'columns=', len(frame.columns))


/kaggle/working/code/backend
-rw-r--r-- 1 root root 126M Aug 31 04:10 data/processed/repositories_combined.csv
-rw-r--r-- 1 root root 49M Aug 31 04:10 data/processed/sljol.csv
-rw-r--r-- 1 root root 286M Aug 31 04:26 data/processed/common/common_publications_final.csv
-rw-r--r-- 1 root root 377M Aug 31 04:30 data/processed/common/common_publications_final_2016_2026_analysis_ready.csv
data/processed/common/common_publications_final.csv rows= 178162 columns= 56
data/processed/common/common_publications_final_2016_2026_analysis_ready.csv rows= 137856 columns= 71


## 8. Build Best-Quality Embeddings

Full text fields, trigrams, larger vocabulary, 512 dimensions, no row limit.

In [9]:
%cd /kaggle/working/code/backend
!make model-embeddings PYTHON=python \
  EMBED_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  EMBED_MAX_FEATURES=100000 \
  EMBED_NGRAM_MAX=3 \
  EMBED_DIM=512
!ls -lh data/models/publication_text_embeddings.parquet data/models/publication_text_embedding_model.joblib data/models/publication_text_embeddings_summary.txt

/kaggle/working/code/backend
python scripts/modeling/generate_publication_text_embeddings.py --input data/processed/common/common_publications_final.csv --output data/models/publication_text_embeddings.parquet --model-output data/models/publication_text_embedding_model.joblib --manifest-output data/models/publication_text_embeddings_manifest.json --summary-output data/models/publication_text_embeddings_summary.txt --text-columns title,abstract,topics,keywords,concepts --metadata-columns record_number,publication_year,title,doi,openalex_id,source_dataset,source_institution_id,source_record_id --embedding-dim 512 --max-features 100000 --min-df 2 --max-df 0.95 --ngram-max 3 
Generated publication text embeddings: rows=177257, dim=512, output=data/models/publication_text_embeddings.parquet
-rw------- 1 root root 395M Aug 31 04:39 data/models/publication_text_embedding_model.joblib
-rw-r--r-- 1 root root 538M Aug 31 04:39 data/models/publication_text_embeddings.parquet
-rw------- 1 root roo

## 9. Train Best-Quality Logistic Regression

In [10]:
%cd /kaggle/working/code/backend
!make train-logreg PYTHON=python \
  LOGREG_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  LOGREG_MAX_FEATURES=100000 \
  LOGREG_NGRAM_MAX=3 \
  LOGREG_MAX_ITER=2000
!cat data/models/logistic_regression_primary_domain_metrics.txt

/kaggle/working/code/backend
python scripts/modeling/train_logistic_regression_classifier.py --input data/processed/common/common_publications_final.csv --label-column primary_domain --text-columns title,abstract,topics,keywords,concepts --model-output data/models/logistic_regression_primary_domain.joblib --metrics-output data/models/logistic_regression_primary_domain_metrics.txt --label-counts-output data/models/logistic_regression_primary_domain_labels.csv --predictions-output data/models/logistic_regression_primary_domain_predictions.csv --manifest-output data/models/logistic_regression_primary_domain_manifest.json --max-features 100000 --min-df 2 --max-df 0.95 --ngram-max 3 --min-class-count 20 --test-size 0.2 --max-iter 2000 
Trained logistic_regression classifier on 52,268 rows.
Classes: 4
Accuracy: 0.8705
Balanced accuracy: 0.8656
Macro F1: 0.8620
Model: data/models/logistic_regression_primary_domain.joblib
Model SHA-256: 2ef9306b6ef19516279b942f253d754a955b4d929479d901ced19a9e5

In [11]:
%cd /kaggle/working/code/backend
!pip install -e . --no-deps

/kaggle/working/code/backend
Obtaining file:///kaggle/working/code/backend
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for research-analytics-framework (pyproject.toml) ... done
  Created wheel for research-analytics-framework: filename=research_analytics_framework-0.1.0-0.editable-py3-none-any.whl size=8228 sha256=fc687a4336ca90278d6e5659da9a83b3ba598594f5696df2918273858b347d5c
  Stored in directory: /tmp/pip-ephem-wheel-cache-k3fzy9l4/wheels/48/6f/98/3205cf08ddaa25af658fb7d96745a6be29b5c675f81653f651
Successfully built research-analytics-framework
  Attempting uninstall: research-analytics-framework
    Found existing installation: research-analytics-framework 0.1.0
    Uninstalling research-analytics-framework-0.1.0:
      Successfully uninstalled research-analytics-framework-0.1.0


## 10. Train Best-Quality Linear SVM

If Kaggle RAM fails, rerun this cell with `--max-features 50000`.

In [12]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_classifier.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,topics,keywords,concepts \
  --ngram-max 3 \
  --max-features 100000 \
  --c-values 0.1,1,10 \
  --cv-folds 3 \
  --class-weight balanced \
  --max-iter 5000 \
  --test-size 0.2
!cat data/models/linear_svm_primary_domain_metrics.txt

/kaggle/working/code/backend
Trained Linear SVM classifier on 52,268 rows.
Classes: 4
Best C: 1.0
CV macro F1: 0.8714
Accuracy: 0.8866
Macro F1: 0.8775
Model: /kaggle/working/code/backend/data/models/linear_svm_primary_domain.joblib
Model SHA-256: 59d843c9e1adf593357b667f3eb9ac0e93cf6798328383fd9d814a2fba63c08e
Metrics: /kaggle/working/code/backend/data/models/linear_svm_primary_domain_metrics.txt
Predictions: /kaggle/working/code/backend/data/models/linear_svm_primary_domain_predictions.csv
Manifest: /kaggle/working/code/backend/data/models/linear_svm_primary_domain_manifest.json
Publication Linear SVM classifier

model_family: linear_svm
input_csv: data/processed/common/common_publications_final.csv
label_column: primary_domain
text_columns: title, abstract, topics, keywords, concepts
input_rows: 178162
usable_rows: 52268
train_rows: 41814
test_rows: 10454
class_count: 4
best_C: 1.0
cv_macro_f1: 0.8714
accuracy: 0.8866
macro_f1: 0.8775
weighted_f1: 0.8866

Class distribution:
Physica

## 11. Train Naive Bayes Baseline


In [13]:
%cd /kaggle/working/code/backend
!make train-nb PYTHON=python \
  NB_INPUT=data/processed/common/common_publications_final.csv \
  NB_LABEL_COLUMN=primary_domain \
  NB_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  NB_ALPHA=1.0 \
  NB_MIN_CLASS_COUNT=20 \
  NB_TEST_SIZE=0.2
!cat data/models/multinomial_nb_primary_domain_metrics.txt


/kaggle/working/code/backend
python -m src.modeling.training --model-family multinomial_nb --input data/processed/common/common_publications_final.csv --label-column primary_domain --text-columns title,abstract,topics,keywords,concepts --alpha 1.0 --min-class-count 20 --test-size 0.2 
Trained multinomial_nb classifier on 52,268 rows.
Classes: 4
Accuracy: 0.8133
Balanced accuracy: 0.7887
Macro F1: 0.7935
Model: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain.joblib
Model SHA-256: 7f5232aaa33622978a89eae832efe12403628212c09f99fdb246ea9bc0328477
Metrics: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain_metrics.txt
Predictions: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain_predictions.csv
Confusion matrix: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain_confusion_matrix.csv
Per-class results: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain_per_class.csv
Manifest: /kaggle/work

## 12. Formal Flat Classifier Comparison

This retrains Logistic Regression and Linear SVM with the same data settings, ranks by macro F1, and copies the winner into `data/models/final/`.


In [14]:
%cd /kaggle/working/code/backend
!python scripts/modeling/compare_classification_models.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,topics,keywords,concepts \
  --max-features 100000 \
  --ngram-max 3 \
  --c-values 0.1,1,10 \
  --cv-folds 3 \
  --class-weight balanced \
  --ranking-metric macro_f1 \
  --test-size 0.2

!cat data/models/classification_comparison/model_comparison.csv
!ls -lh data/models/final


/kaggle/working/code/backend
Compared 3 classification model families.
Ranking metric: macro_f1
Best model: linear_svm
Comparison: /kaggle/working/code/backend/data/models/classification_comparison/model_comparison.csv
Manifest: /kaggle/working/code/backend/data/models/classification_comparison/model_comparison_manifest.json
Final field model: /kaggle/working/code/backend/data/models/final/publication_field_classifier.joblib
Final manifest: /kaggle/working/code/backend/data/models/final/publication_field_classifier_manifest.json




total 8.2M
-rw------- 1 root root 8.2M Aug 31 04:56 publication_field_classifier.joblib
-rw------- 1 root root  691 Aug 31 04:56 publication_field_classifier_manifest.json
-rw------- 1 root root 1.1K Aug 31 04:56 publication_field_classifier_metrics.txt


## 13. Evaluate Prediction Files Together


In [15]:
%cd /kaggle/working/code/backend
!make evaluate-models PYTHON=python \
  EVAL_PREDICTIONS="--predictions-csv data/models/classification_comparison/multinomial_nb_primary_domain_predictions.csv --predictions-csv data/models/classification_comparison/logistic_regression_primary_domain_predictions.csv --predictions-csv data/models/classification_comparison/linear_svm_primary_domain_predictions.csv" \
  EVAL_OUTPUT_DIR=data/models/evaluation
!find data/models/evaluation -maxdepth 2 -type f -print | sort


/kaggle/working/code/backend
python -m src.modeling.evaluation --predictions-csv data/models/classification_comparison/multinomial_nb_primary_domain_predictions.csv --predictions-csv data/models/classification_comparison/logistic_regression_primary_domain_predictions.csv --predictions-csv data/models/classification_comparison/linear_svm_primary_domain_predictions.csv --output-dir data/models/evaluation 
Evaluation: multinomial_nb_primary_domain

rows: 10454
class_count: 4
accuracy: 0.8188
balanced_accuracy: 0.7913
macro_precision: 0.8101
macro_recall: 0.7913
macro_f1: 0.7981
weighted_f1: 0.8164

Per-class results:
label                        support    prec  recall      f1  most confused with
Physical Sciences               3435   0.839   0.840   0.839  Social Sciences (353)
Social Sciences                 3050   0.816   0.883   0.848  Physical Sciences (215)
Health Sciences                 2509   0.819   0.835   0.827  Social Sciences (158)
Life Sciences                   1460   0.76

## 14. Train Hierarchical Field -> Subfield Linear SVM


In [16]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_hierarchical.py \
  --input data/processed/common/common_publications_final.csv \
  --field-column primary_field \
  --subfield-column primary_subfield \
  --text-columns title,abstract,topics,keywords,concepts \
  --max-features 100000 \
  --ngram-max 3 \
  --c-value 1.0 \
  --class-weight balanced \
  --max-iter 5000 \
  --predict-output data/models/linear_svm_hierarchical_predictions.csv
from pathlib import Path

metrics_candidates = [
    Path("data/models/linear_svm_hierarchical_metrics.txt"),
    Path("data/models/linear_svm_hierarchical_field_metrics.txt"),
]
metrics_path = next((path for path in metrics_candidates if path.exists()), None)
if metrics_path is None:
    raise FileNotFoundError(
        "Missing hierarchical metrics file. Checked: "
        + ", ".join(str(path) for path in metrics_candidates)
    )
print(metrics_path.read_text())
!ls -lh data/models/linear_svm_hierarchical*


/kaggle/working/code/backend
Trained hierarchical Linear SVM on 52,268 rows.
Field classes: 26
Subfield models: 26
Field accuracy: 0.8021
Field macro F1: 0.7456
Subfield mean accuracy: 0.9018
Subfield mean macro F1: 0.8136
Field model: /kaggle/working/code/backend/data/models/linear_svm_hierarchical_field.joblib
Subfield models: /kaggle/working/code/backend/data/models/linear_svm_hierarchical_subfields.joblib
Metrics: /kaggle/working/code/backend/data/models/linear_svm_hierarchical_metrics.txt
Manifest: /kaggle/working/code/backend/data/models/linear_svm_hierarchical_manifest.json
Predictions written to: data/models/linear_svm_hierarchical_predictions.csv
Publication hierarchical Linear SVM (field → subfield)

model_family: linear_svm_hierarchical
input_csv: data/processed/common/common_publications_final.csv
taxonomy_json: /kaggle/working/code/backend/category_hierarchy.json
field_column: primary_field
subfield_column: primary_subfield
text_columns: title, abstract, topics, keywords, 

## 15. Run NMF Topic Modeling And Evaluation


In [17]:
%cd /kaggle/working/code/backend

!python scripts/modeling/run_nmf_topic_modeling.py \
  --data data/processed/common/common_publications_final.csv \
  --output-dir data/processed/common/nmf \
  --k-range 8 10 12 \
  --n-words 15 \
  --naming-words 3 \
  --max-iter 1000 \
  --text-columns title abstract topics keywords concepts

!find data/processed/common/nmf -maxdepth 1 -type f -print | sort

!cat data/processed/common/nmf/nmf_k_sweep_evaluation.csv


/kaggle/working/code/backend
Loading data/processed/common/common_publications_final.csv ...
Shape: (178162, 56)

Cleaning check: 7900 of 178162 rows contain Tamil/Sinhala-script characters; 4414 rows contain metadata-boilerplate phrases (abstract available / editorial / etc.).
clean=True — stripping these at the token/phrase level (rows are never dropped).

No --k given, sweeping k in [8, 10, 12] ...
k=  8  coherence_cv=0.8294  diversity=0.908  redundancy=0.016  recon_err=414.5850
k= 10  coherence_cv=0.8461  diversity=0.900  redundancy=0.014  recon_err=414.2297
k= 12  coherence_cv=0.8165  diversity=0.894  redundancy=0.012  recon_err=413.8633

Best k by coherence: 10
Cleaning report: 7900 of 178162 rows had non-Latin chars, 4414 rows had boilerplate phrases. clean=True
895 rows had text before cleaning but are empty after it (e.g. a title/abstract that was entirely Tamil/boilerplate) and are excluded from the NMF fit.

k=10  coherence_cv=0.8461191962816036  diversity=0.900  redundancy=

In [18]:
%cd /kaggle/working/code/backend
import shutil
from pathlib import Path

aliases = {
    "data/models/linear_svm_hierarchical_subfield.joblib": "data/models/linear_svm_hierarchical_subfields.joblib",
    "data/models/linear_svm_hierarchical_field_metrics.txt": "data/models/linear_svm_hierarchical_metrics.txt",
}

for source, target in aliases.items():
    source_path = Path(source)
    target_path = Path(target)
    if source_path.exists() and not target_path.exists():
        shutil.copy2(source_path, target_path)
        print(f"Aliased {source} -> {target}")

!ls -lh data/models/linear_svm_hierarchical*

/kaggle/working/code/backend
-rw------- 1 root root  25M Aug 31 04:59 data/models/linear_svm_hierarchical_field.joblib
-rw------- 1 root root   42 Aug 31 04:59 data/models/linear_svm_hierarchical_field_predictions.csv
-rw------- 1 root root  693 Aug 31 04:59 data/models/linear_svm_hierarchical_labels.csv
-rw------- 1 root root 8.3K Aug 31 04:59 data/models/linear_svm_hierarchical_manifest.json
-rw------- 1 root root 2.7K Aug 31 04:59 data/models/linear_svm_hierarchical_metrics.txt
-rw-r--r-- 1 root root 295M Aug 31 05:02 data/models/linear_svm_hierarchical_predictions.csv
-rw------- 1 root root 115M Aug 31 04:59 data/models/linear_svm_hierarchical_subfields.joblib


## 16. Verify All Modeling Outputs


In [19]:
%cd /kaggle/working/code/backend
from pathlib import Path

required = [
    "data/models/publication_text_embeddings.parquet",
    "data/models/publication_text_embedding_model.joblib",
    "data/models/logistic_regression_primary_domain.joblib",
    "data/models/logistic_regression_primary_domain_metrics.txt",
    "data/models/multinomial_nb_primary_domain.joblib",
    "data/models/multinomial_nb_primary_domain_metrics.txt",
    "data/models/linear_svm_primary_domain.joblib",
    "data/models/linear_svm_primary_domain_metrics.txt",
    "data/models/classification_comparison/model_comparison.csv",
    "data/models/final/publication_field_classifier.joblib",
    "data/models/linear_svm_hierarchical_field.joblib",
    "data/models/linear_svm_hierarchical_subfields.joblib",
    "data/models/linear_svm_hierarchical_metrics.txt",
    "data/processed/common/nmf/nmf_k_sweep_evaluation.csv",
    "data/reports/openalex_lk_affiliation_audit/verified_lk_authorships.csv",
    "data/reports/openalex_lk_affiliation_audit/lk_affiliation_manual_review.csv",
    "data/reports/openalex_lk_affiliation_audit/lk_affiliation_audit_summary.json",
    "data/reports/openalex_lk_affiliation_audit/lk_affiliation_audit_report.md",
]

missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing modeling outputs:\n" + "\n".join(missing))

for p in required:
    path = Path(p)
    print(f"OK {p} ({path.stat().st_size / (1024*1024):.2f} MB)")


/kaggle/working/code/backend
OK data/models/publication_text_embeddings.parquet (537.63 MB)
OK data/models/publication_text_embedding_model.joblib (394.91 MB)
OK data/models/logistic_regression_primary_domain.joblib (7.55 MB)
OK data/models/logistic_regression_primary_domain_metrics.txt (0.00 MB)
OK data/models/multinomial_nb_primary_domain.joblib (5.13 MB)
OK data/models/multinomial_nb_primary_domain_metrics.txt (0.00 MB)
OK data/models/linear_svm_primary_domain.joblib (8.15 MB)
OK data/models/linear_svm_primary_domain_metrics.txt (0.00 MB)
OK data/models/classification_comparison/model_comparison.csv (0.00 MB)
OK data/models/final/publication_field_classifier.joblib (8.15 MB)
OK data/models/linear_svm_hierarchical_field.joblib (24.94 MB)
OK data/models/linear_svm_hierarchical_subfields.joblib (114.99 MB)
OK data/models/linear_svm_hierarchical_metrics.txt (0.00 MB)
OK data/processed/common/nmf/nmf_k_sweep_evaluation.csv (0.00 MB)
OK data/reports/openalex_lk_affiliation_audit/verified_

## 17. Zip Outputs For Download


In [20]:
%cd /kaggle/working/code/backend
!rm -f /kaggle/working/researchlanka-kaggle-outputs.zip
!zip -r /kaggle/working/researchlanka-kaggle-outputs.zip data/processed data/models data/reports
!ls -lh /kaggle/working/researchlanka-kaggle-outputs.zip


/kaggle/working/code/backend
  adding: data/processed/ (stored 0%)
  adding: data/processed/repositories/ (stored 0%)
  adding: data/processed/repositories/pdn.jsonl (deflated 70%)
  adding: data/processed/repositories/ou.jsonl (deflated 78%)
  adding: data/processed/repositories/nsf.jsonl (deflated 81%)
  adding: data/processed/repositories/uom.jsonl (deflated 72%)
  adding: data/processed/repositories/seu.jsonl (deflated 77%)
  adding: data/processed/repositories/sliit.jsonl (deflated 71%)
  adding: data/processed/repositories/jfn_medicine.jsonl (deflated 89%)
  adding: data/processed/repositories/busl.jsonl (deflated 85%)
  adding: data/processed/repositories/cmb.jsonl (deflated 71%)
  adding: data/processed/repositories/ruh.jsonl (deflated 73%)
  adding: data/processed/repositories/jfn_research.jsonl (deflated 78%)
  adding: data/processed/sljol.csv (deflated 82%)
  adding: data/processed/repositories_combined.csv (deflated 69%)
  adding: data/processed/crossref/ (stored 0%)
  addi

Download these files from the Kaggle output panel:


In [21]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/researchlanka-kaggle-outputs.zip"))


/kaggle/working/researchlanka-kaggle-outputs.zip

## 18. Optional: Create Models-Only Zip


In [22]:
import os
import zipfile
from IPython.display import FileLink, display

source_zip = "/kaggle/working/researchlanka-kaggle-outputs.zip"
models_zip = "/kaggle/working/researchlanka-models-only.zip"

with zipfile.ZipFile(source_zip, "r") as src:
    model_files = [
        name for name in src.namelist()
        if name.startswith("data/models/") and not name.endswith("/")
    ]
    print("Model files found:", len(model_files))
    with zipfile.ZipFile(models_zip, "w", zipfile.ZIP_DEFLATED) as dst:
        for name in model_files:
            dst.writestr(name, src.read(name))

print("Created:", models_zip)
print("Size MB:", round(os.path.getsize(models_zip) / (1024**2), 2))
display(FileLink(models_zip))


Model files found: 61
Created: /kaggle/working/researchlanka-models-only.zip
Size MB: 1078.92


/kaggle/working/researchlanka-models-only.zip